### Making the dataset file structure compatible with the SDG Engine's

Convert the Homebrew dataset to an easily loadable format for Hugging Face's datasets library.

In [14]:
import os
from typing import List, Dict
import shutil
import json

dataset_jsonl: List[Dict] = []
dataset_path = "/Users/federico/Documents/personal/projects/sdg-engine-files/homebrew-scenes/val_primesense/000003"
target_dataset_path = "/Users/federico/Documents/personal/code/sdg-engine/tutorials/real-bop-homebrew-scene-3/val"

# if the target dataset path does not exist, create it
if not os.path.exists(target_dataset_path):
    os.makedirs(target_dataset_path)

# Path to the rgb images
rgb_images = os.listdir(os.path.join(dataset_path, "rgb"))

# load the scene_gt_info.json file
scene_gt_info = json.load(
    open(os.path.join(dataset_path, "scene_gt_info.json"))
)

# Iterate over the rgb images
for image in rgb_images:
    annotation_dict = {}
    # Copy the image to the target dataset path
    img_new_path = os.path.join(target_dataset_path, image)
    shutil.copy(
        os.path.join(dataset_path, "rgb", image),
        img_new_path,
    )
    # Collect the object's bounding box from the scene_gt_info.json file
    image_id = image.split(".")[0].lstrip("0") or "0"
    if len(scene_gt_info[image_id]) == 1:
        image_bbox = [scene_gt_info[image_id][0]["bbox_obj"]]
    else:
        image_bbox = [annotation["bbox_obj"] for annotation in scene_gt_info[image_id]]

    # Populate the annotation dictionary
    annotation_dict["image_id"] = image_id
    annotation_dict["file_name"] = image
    annotation_dict["objects"] = {
        "bbox": image_bbox,
        "bbox_id": [int(image_id)*1000+i for i in range(len(image_bbox))],
        "area": [image_bbox[i][2]*image_bbox[i][3] for i in range(len(image_bbox))],
        "category": [i for i in range(len(image_bbox))],
    }

    dataset_jsonl.append(annotation_dict)

# Finally, save the dataset_jsonl to a file
with open(f"{target_dataset_path}/metadata.jsonl", "w") as f:
    for annotation in dataset_jsonl:
        f.write(json.dumps(annotation) + "\n")

Now load them into Hugging Face datasets.

In [15]:
from datasets import load_dataset

DATASET_PATH = "/Users/federico/Documents/personal/code/sdg-engine/tutorials/real-bop-homebrew-scene-3"
dataset = load_dataset("imagefolder", data_dir=DATASET_PATH)
print(f"Dataset loaded: \n{dataset}")

Generating validation split: 340 examples [00:00, 18135.45 examples/s]

Dataset loaded: 
DatasetDict({
    validation: Dataset({
        features: ['image', 'image_id', 'objects'],
        num_rows: 340
    })
})


Let's push this validation dataset to HuggingFace.

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN")

dataset.push_to_hub(
    "federicoarenas-ai/real-bop-homebrew-scene-3",
    token="",
    private=True
)

Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.86s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/federicoarenas-ai/real-bop-homebrew-scene-3/commit/965e03f3f31235981b81aac5a69e46bd4d66fead', commit_message='Upload dataset', commit_description='', oid='965e03f3f31235981b81aac5a69e46bd4d66fead', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/federicoarenas-ai/real-bop-homebrew-scene-3', endpoint='https://huggingface.co', repo_type='dataset', repo_id='federicoarenas-ai/real-bop-homebrew-scene-3'), pr_revision=None, pr_num=None)

### TODO: Push synthetic datasets to HF

In [22]:
DATASET_PATH = "/Users/federico/Documents/personal/projects/code/vit-sdg-engine/hf-tudl-rgb-dataset-render/"
dataset_render = load_dataset("imagefolder", data_dir=DATASET_PATH)
print(f"Dataset loaded: \n{dataset_render}")

Generating train split: 5481 examples [00:00, 10415.32 examples/s]


Dataset loaded: 
DatasetDict({
    train: Dataset({
        features: ['image_id', 'image', 'objects'],
        num_rows: 5481
    })
})


Finally, lets push them to the Hugging Face Hub.

In [ ]:
dataset_render.push_to_hub(
    "federicoarenas-ai/bop-tudl-rgb-dataset-render",
    token=HF_TOKEN,
)

Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.42s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/federicoarenas-ai/bop-tudl-rgb-dataset-render/commit/e88d060fbdd3e403b42c2917c374a7ce5562920b', commit_message='Upload dataset', commit_description='', oid='e88d060fbdd3e403b42c2917c374a7ce5562920b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/federicoarenas-ai/bop-tudl-rgb-dataset-render', endpoint='https://huggingface.co', repo_type='dataset', repo_id='federicoarenas-ai/bop-tudl-rgb-dataset-render'), pr_revision=None, pr_num=None)